# Fase 2.2 — Modelo LSTM

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook:
1. Loads the feature CSVs produced in Fase 1
2. Scales features with MinMaxScaler (fitted on train)
3. Builds sliding-window sequences (lookback=168h, horizon=24h)
4. Trains a 2-layer stacked LSTM with EarlyStopping
5. Evaluates on the test set (stride=24 → non-overlapping daily blocks)
6. Saves predictions and metrics

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

## 1 · Load data

In [ ]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')

# Feature columns — all numeric except datetime
FEATURE_COLS = [c for c in df_train.columns if c != 'datetime']
TARGET_COL   = 'demand_mw'
print(f'Features: {len(FEATURE_COLS)}')

## 2 · Normalise with MinMaxScaler (fit on train)

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df_train[FEATURE_COLS])

train_scaled = scaler.transform(df_train[FEATURE_COLS])
val_scaled   = scaler.transform(df_val[FEATURE_COLS])
test_scaled  = scaler.transform(df_test[FEATURE_COLS])

# Target column index (needed for inverse-transform later)
target_idx = FEATURE_COLS.index(TARGET_COL)

train_feat = train_scaled
train_tgt  = train_scaled[:, target_idx]

val_feat   = val_scaled
val_tgt    = val_scaled[:, target_idx]

test_feat  = test_scaled
test_tgt   = test_scaled[:, target_idx]

print(f'Scaled arrays — train: {train_feat.shape}  val: {val_feat.shape}  test: {test_feat.shape}')

## 3 · Build sliding-window sequences

In [ ]:
LOOKBACK = 168   # 1 week in hours
HORIZON  = 24    # 24h ahead forecast

# Training: stride=1 (dense overlapping windows for maximum data)
X_train, y_train = du.make_sequences(train_feat, train_tgt, LOOKBACK, HORIZON, stride=1)

# Validation: stride=1
X_val, y_val = du.make_sequences(val_feat, val_tgt, LOOKBACK, HORIZON, stride=1)

# Test: stride=24 → non-overlapping daily blocks
X_test, y_test = du.make_sequences(test_feat, test_tgt, LOOKBACK, HORIZON, stride=HORIZON)

print(f'X_train {X_train.shape}  y_train {y_train.shape}')
print(f'X_val   {X_val.shape}    y_val   {y_val.shape}')
print(f'X_test  {X_test.shape}   y_test  {y_test.shape}')

## 4 · Build LSTM model

In [ ]:
def build_lstm(input_shape, horizon):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape,
             dropout=0.2, recurrent_dropout=0.0),
        Dropout(0.2),
        LSTM(64, return_sequences=False,
             dropout=0.2, recurrent_dropout=0.0),
        Dropout(0.2),
        Dense(horizon, activation='linear'),
    ])
    model.compile(
        optimizer = Adam(learning_rate=1e-3),
        loss      = 'mse',
        metrics   = ['mae'],
    )
    return model

model = build_lstm(
    input_shape = (LOOKBACK, X_train.shape[2]),
    horizon     = HORIZON,
)
model.summary()

## 5 · Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint('saved_models/lstm_best.keras', monitor='val_loss',
                    save_best_only=True, verbose=0),
]

print('Training…')
t0 = time.time()

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = 150,
    batch_size      = 128,
    callbacks       = callbacks,
    shuffle         = False,
    verbose         = 1,
)

train_time = time.time() - t0
print(f'\nTraining completed in {train_time:.1f}s  ({train_time/60:.1f} min)')
print(f'Best val_loss: {min(history.history["val_loss"]):.6f}')

In [ ]:
du.plot_training_history(
    history,
    title     = 'LSTM training',
    save_path = 'data/fig_lstm_training.png',
)

## 6 · Inference on test set

In [ ]:
t0 = time.time()
y_pred_scaled = model.predict(X_test, verbose=0)
inference_time = time.time() - t0
print(f'Inference time: {inference_time:.2f}s  ({inference_time*1000/len(X_test):.1f}ms/sample)')

# Inverse-transform predictions and actuals back to MW
# We only scaled one column; use scaler's min/max for that column
t_min = scaler.data_min_[target_idx]
t_max = scaler.data_max_[target_idx]

y_pred_mw = y_pred_scaled * (t_max - t_min) + t_min
y_true_mw = y_test        * (t_max - t_min) + t_min

# Flatten for metric computation
y_pred_flat = y_pred_mw.ravel()
y_true_flat = y_true_mw.ravel()

# Corresponding datetimes (stride=24, first date of each window + LOOKBACK offset)
test_datetimes = df_test['datetime'].values
dates_flat = np.concatenate([
    test_datetimes[LOOKBACK + i * HORIZON : LOOKBACK + i * HORIZON + HORIZON]
    for i in range(len(X_test))
])
print(f'Forecast points: {len(y_pred_flat):,}')

## 7 · Metrics

In [ ]:
metrics = du.compute_metrics(y_true_flat, y_pred_flat, label='LSTM')
metrics['train_s']     = train_time
metrics['inference_s'] = inference_time

## 8 · Plots

In [ ]:
du.plot_predictions(
    y_true_flat, y_pred_flat,
    title     = 'LSTM — actual vs predicted (test set)',
    dates     = pd.to_datetime(dates_flat),
    save_path = 'data/fig_lstm_test.png',
)

In [ ]:
# Zoom into first week
idx = slice(0, 168)
du.plot_predictions(
    y_true_flat[idx], y_pred_flat[idx],
    title     = 'LSTM — first week of test set',
    dates     = pd.to_datetime(dates_flat[idx]),
    save_path = 'data/fig_lstm_week.png',
)

In [ ]:
# Error distribution
errors = y_true_flat - y_pred_flat
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(errors, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('LSTM — residual distribution')
axes[0].set_xlabel('Error (MW)')

axes[1].plot(pd.to_datetime(dates_flat), errors, linewidth=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('LSTM — residuals over time')
axes[1].set_ylabel('Error (MW)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('data/fig_lstm_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Save predictions and metrics

In [ ]:
import json

df_preds = pd.DataFrame({
    'datetime': dates_flat,
    'y_true':   y_true_flat,
    'y_pred':   y_pred_flat,
})
df_preds.to_csv('data/predictions_lstm.csv', index=False)

with open('data/metrics_lstm.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved {len(df_preds):,} rows → data/predictions_lstm.csv')
print(f'Saved metrics → data/metrics_lstm.json')
print(metrics)

## Summary

| Metric | Value |
|--------|-------|
| MAPE   | … % |
| RMSE   | … MW |
| MAE    | … MW |
| Training time | … s |
| Inference time | … s |

Next step → `fase2_3_ttm.ipynb`